# Baseline improvement by Species-specific modeling and One Hot Encoding

Species specific modeling for:
- Amoxicillin_Clavulanic_acid
- Levofloxacin
- Ciprofloxacin

One-Hot encoding of `species_id`

Init path and imports

In [26]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
data_folder = '/content/drive/MyDrive/ML/Kaggle'
os.chdir(data_folder)

Load CSVs

In [29]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
sub   = pd.read_csv("sample_submission.csv")

print(train.shape)
print(test.shape)
print(sub.shape)
train.head(2)


(3360, 6010)
(1000, 6002)
(1000, 9)


,sample_id,species_id,maldi_feature_0,maldi_feature_1,maldi_feature_2,maldi_feature_3,maldi_feature_4,maldi_feature_5,maldi_feature_6,maldi_feature_7,...,maldi_feature_5998,maldi_feature_5999,Ampicillin,Levofloxacin,Ciprofloxacin,Imipenem,Amoxicillin_Clavulanic_acid,Ertapenem,Cefotaxime,Cefuroxime
0,SAMPLE_00000,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,SAMPLE_00001,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


Separate feature columns and target columns

In [16]:
TARGETS = [
    "Ampicillin",
    "Levofloxacin",
    "Ciprofloxacin",
    "Imipenem",
    "Amoxicillin_Clavulanic_acid",
    "Ertapenem",
    "Cefotaxime",
    "Cefuroxime",
]

# Changed for OHE version
feature_cols = (
    [c for c in train.columns if c.startswith("sp_")] +
    [c for c in train.columns if c.startswith("maldi_feature_")]
)

print([c for c in feature_cols if c.startswith("sp_")])
print("Number of features:", len(feature_cols))
print("First 5 features:", feature_cols[:5])
print("Last 5 features:", feature_cols[-5:])


['sp_0', 'sp_1', 'sp_2', 'sp_3']
Number of features: 6004
First 5 features: ['sp_0', 'sp_1', 'sp_2', 'sp_3', 'maldi_feature_0']
Last 5 features: ['maldi_feature_5995', 'maldi_feature_5996', 'maldi_feature_5997', 'maldi_feature_5998', 'maldi_feature_5999']


# SPECIES-SPECIFIC MODELING CHECKS

Check if species-specific modeling is viable

In [14]:
t = "Levofloxacin"

labeled = train.dropna(subset=[t]).copy()

counts = (
    labeled
    .groupby("species_id")[t]
    .count()
    .rename("labeled_samples")
)

print("Levofloxacin")
print(counts)

print("-----------------------")

t = "Ciprofloxacin"

labeled = train.dropna(subset=[t]).copy()

counts = (
    labeled
    .groupby("species_id")[t]
    .count()
    .rename("labeled_samples")
)

print("Ciprofloxacin")
print(counts)

Levofloxacin
species_id
0     533
1     928
2     413
3    1397
Name: labeled_samples, dtype: int64
-----------------------
Ciprofloxacin
species_id
0     531
1     928
2     413
3    1402
Name: labeled_samples, dtype: int64


Therefore the training strategy for both antibiotics will be:
1. Split labeled data by species_id

2. Train 4 separate XGBoost models:

    - one for species 0

    - one for species 1

    - one for species 2

    - one for species 3

In [17]:
def make_xgb(seed=42):
    return XGBClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        n_jobs=-1,
        random_state=seed,
    )

In [18]:
labeled = train.dropna(subset=[t]).copy()
X_all = labeled[feature_cols]
y_all = labeled[t].astype(int)

print("Total labeled:", len(labeled))
print(labeled["species_id"].value_counts().sort_index())


Total labeled: 3274


KeyError: 'species_id'

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_global = np.zeros(len(labeled), dtype=float)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_all), start=1):
    X_tr, X_va = X_all.iloc[tr_idx], X_all.iloc[va_idx]
    y_tr, y_va = y_all.iloc[tr_idx], y_all.iloc[va_idx]

    model = make_xgb(seed=42 + fold)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    oof_global[va_idx] = model.predict_proba(X_va)[:, 1]

auc_global = roc_auc_score(y_all, oof_global)
print("Global OOF AUC:", auc_global)


Global OOF AUC: 0.7016874475095289


In [ ]:
maldi_only = [c for c in feature_cols if c.startswith("maldi_feature_")]

oof_hybrid = oof_global.copy()  # global fallback

for sp in [0, 1, 2]:
    sub = labeled[labeled["species_id"] == sp].copy()
    X_sp = sub[maldi_only]
    y_sp = sub[t].astype(int)

    skf_sp = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_sp = np.zeros(len(sub), dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(skf_sp.split(X_sp, y_sp), start=1):
        X_tr, X_va = X_sp.iloc[tr_idx], X_sp.iloc[va_idx]
        y_tr, y_va = y_sp.iloc[tr_idx], y_sp.iloc[va_idx]

        model = make_xgb(seed=1000 + sp * 10 + fold)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

        oof_sp[va_idx] = model.predict_proba(X_va)[:, 1]

    pos = labeled.index.get_indexer(sub.index)
    oof_hybrid[pos] = oof_sp

    print(f"species {sp} OOF AUC:", roc_auc_score(y_sp, oof_sp))


species 0 OOF AUC: 0.5851016588955499
species 1 OOF AUC: 0.6961457924355451
species 2 OOF AUC: 0.5951360617098779


In [ ]:
auc_hybrid = roc_auc_score(y_all, oof_hybrid)
print("Global OOF AUC :", auc_global)
print("Hybrid OOF AUC :", auc_hybrid)
print("Delta          :", auc_hybrid - auc_global)


Global OOF AUC : 0.7016874475095289
Hybrid OOF AUC : 0.7135628916596678
Delta          : 0.011875444150138859


Apparently seems like the individual species models will work better alone.

# Full pipeline

In [30]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# -----------------------------
# CONFIG
# -----------------------------
TARGETS = [
    "Ampicillin",
    "Levofloxacin",
    "Ciprofloxacin",
    "Imipenem",
    "Amoxicillin_Clavulanic_acid",
    "Ertapenem",
    "Cefotaxime",
    "Cefuroxime",
]

SPECIES_SPECIFIC_TARGETS = {"Levofloxacin", "Ciprofloxacin"}          # full species-specific
HYBRID_TARGET = "Amoxicillin_Clavulanic_acid"                         # hybrid: sp 0/1/2 + global fallback for sp3
HYBRID_SPECIES = [0, 1, 2]                                             # species to specialize for HYBRID_TARGET

def make_xgb(seed=42):
    return XGBClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",   # GPU not available in your current xgboost build
        n_jobs=-1,
        random_state=seed,
    )

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# -----------------------------
# OHE SETUP (species_id -> sp_0..sp_3)
# -----------------------------
train_oh = pd.get_dummies(train, columns=["species_id"], prefix="sp")
test_oh  = pd.get_dummies(test,  columns=["species_id"], prefix="sp")

# ensure identical columns in train/test after OHE
train_oh, test_oh = train_oh.align(test_oh, join="left", axis=1, fill_value=0)

sp_cols    = [c for c in train_oh.columns if c.startswith("sp_")]
maldi_cols = [c for c in train_oh.columns if c.startswith("maldi_feature_")]

# features for GLOBAL models (OHE species + MALDI)
feature_cols_oh = sp_cols + maldi_cols

# features for SPECIES-SPECIFIC models (MALDI only)
maldi_only = maldi_cols

# recover numeric species for routing
train_oh["species_id_num"] = train_oh[sp_cols].values.argmax(axis=1)
test_oh["species_id_num"]  = test_oh[sp_cols].values.argmax(axis=1)

DF_TRAIN, DF_TEST = train_oh, test_oh
species_col = "species_id_num"

# outputs
oof_all = train[["sample_id"]].copy()                       # keep original train ids
test_pred = pd.DataFrame({"sample_id": test["sample_id"]})  # submission frame
aucs = {}

# -----------------------------
# HELPERS
# -----------------------------
def train_species_specific_full(train_df, test_df, target, maldi_cols, species_col, skf, make_xgb, seed_base=6000):
    """
    Full species-specific training:
      - OOF: CV done within each species
      - TEST: one final model per species; route predictions by species
    Uses MALDI-only features.
    """
    labeled = train_df.dropna(subset=[target]).copy()
    oof = np.zeros(len(labeled), dtype=float)
    test_pred_col = np.zeros(len(test_df), dtype=float)

    for sp in sorted(labeled[species_col].unique()):
        sub = labeled[labeled[species_col] == sp].copy()
        Xs = sub[maldi_cols]
        ys = sub[target].astype(int)

        oof_sp = np.zeros(len(sub), dtype=float)
        for fold, (tr_idx, va_idx) in enumerate(skf.split(Xs, ys), start=1):
            X_tr, X_va = Xs.iloc[tr_idx], Xs.iloc[va_idx]
            y_tr, y_va = ys.iloc[tr_idx], ys.iloc[va_idx]

            model = make_xgb(seed=seed_base + int(sp) * 10 + fold)
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            oof_sp[va_idx] = model.predict_proba(X_va)[:, 1]

        # write back into labeled positions
        pos = labeled.index.get_indexer(sub.index)
        oof[pos] = oof_sp

        # final model for test routing
        model_final = make_xgb(seed=seed_base + 100 + int(sp))
        model_final.fit(Xs, ys, verbose=False)

        mask_test = (test_df[species_col].values == sp)
        if mask_test.any():
            test_pred_col[mask_test] = model_final.predict_proba(
                test_df.loc[mask_test, maldi_cols]
            )[:, 1]

    auc = roc_auc_score(labeled[target].astype(int), oof)
    return labeled, oof, test_pred_col, auc

def train_hybrid_amox(train_df, test_df, target, feature_cols_global, maldi_cols, species_col, skf, make_xgb,
                      hybrid_species=(0,1,2), seed_base=8000):
    """
    Hybrid for Amoxicillin_Clavulanic_acid:
      - Global model provides fallback predictions everywhere (esp. rare species)
      - Species-specific CV overwrites OOF for selected species (hybrid_species)
      - TEST: start with global predictions, overwrite selected species using per-species final models
    """
    labeled = train_df.dropna(subset=[target]).copy()
    y = labeled[target].astype(int)

    # (A) Global CV predictions (fallback baseline)
    Xg = labeled[feature_cols_global]
    oof_global = np.zeros(len(labeled), dtype=float)
    test_global_folds = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(Xg, y), start=1):
        X_tr, X_va = Xg.iloc[tr_idx], Xg.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = make_xgb(seed=42 + fold)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

        oof_global[va_idx] = model.predict_proba(X_va)[:, 1]
        test_global_folds.append(model.predict_proba(test_df[feature_cols_global])[:, 1])

    oof_hybrid = oof_global.copy()

    # (B) Species-specific CV overwrite for chosen species (MALDI-only)
    for sp in hybrid_species:
        sub = labeled[labeled[species_col] == sp].copy()
        Xs = sub[maldi_cols]
        ys = sub[target].astype(int)

        oof_sp = np.zeros(len(sub), dtype=float)
        for fold, (tr_idx, va_idx) in enumerate(skf.split(Xs, ys), start=1):
            X_tr, X_va = Xs.iloc[tr_idx], Xs.iloc[va_idx]
            y_tr, y_va = ys.iloc[tr_idx], ys.iloc[va_idx]

            model = make_xgb(seed=seed_base + int(sp) * 10 + fold)
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

            oof_sp[va_idx] = model.predict_proba(X_va)[:, 1]

        pos = labeled.index.get_indexer(sub.index)
        oof_hybrid[pos] = oof_sp

    auc = roc_auc_score(y, oof_hybrid)

    # (C) TEST predictions: start with global average, overwrite chosen species with per-species final models
    test_pred_col = np.mean(np.vstack(test_global_folds), axis=0)

    for sp in hybrid_species:
        train_sp = labeled[labeled[species_col] == sp].copy()
        X_train_sp = train_sp[maldi_cols]
        y_train_sp = train_sp[target].astype(int)

        model_sp = make_xgb(seed=seed_base + 1000 + int(sp))
        model_sp.fit(X_train_sp, y_train_sp, verbose=False)

        mask_test = (test_df[species_col].values == sp)
        if mask_test.any():
            test_pred_col[mask_test] = model_sp.predict_proba(
                test_df.loc[mask_test, maldi_cols]
            )[:, 1]

    return labeled, oof_hybrid, test_pred_col, auc

# -----------------------------
# TRAIN LOOP
# -----------------------------
for t in TARGETS:
    print(f"\nTraining target: {t}")

    # 1) Full species-specific: Levo + Cipro (MALDI only)
    if t in SPECIES_SPECIFIC_TARGETS:
        labeled, oof_vec, test_col, auc = train_species_specific_full(
            train_df=DF_TRAIN,
            test_df=DF_TEST,
            target=t,
            maldi_cols=maldi_only,
            species_col=species_col,
            skf=skf,
            make_xgb=make_xgb,
            seed_base=6000,
        )
        print(f"  OOF AUC (Species-specific {t}): {auc:.5f}   (labeled={len(labeled)})")

    # 2) Hybrid: Amox/Clav (sp 0/1/2 specialized, global fallback for others)
    elif t == HYBRID_TARGET:
        labeled, oof_vec, test_col, auc = train_hybrid_amox(
            train_df=DF_TRAIN,
            test_df=DF_TEST,
            target=t,
            feature_cols_global=feature_cols_oh,  # sp_* + maldi_*
            maldi_cols=maldi_only,                # maldi_* only
            species_col=species_col,
            skf=skf,
            make_xgb=make_xgb,
            hybrid_species=HYBRID_SPECIES,
            seed_base=8000,
        )
        print(f"  OOF AUC (Hybrid {t}): {auc:.5f}   (labeled={len(labeled)})")

    # 3) Default global: all other antibiotics (sp_* + MALDI)
    else:
        labeled = DF_TRAIN.dropna(subset=[t]).copy()
        y = labeled[t].astype(int)
        X = labeled[feature_cols_oh]

        oof_vec = np.zeros(len(labeled), dtype=float)
        test_folds = []

        for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

            model = make_xgb(seed=42 + fold)
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

            pred_va = model.predict_proba(X_va)[:, 1]
            oof_vec[va_idx] = pred_va

            test_folds.append(model.predict_proba(DF_TEST[feature_cols_oh])[:, 1])

        test_col = np.mean(np.vstack(test_folds), axis=0)
        auc = roc_auc_score(y, oof_vec)
        print(f"  OOF AUC ({t}): {auc:.5f}   (labeled={len(labeled)})")

    # store results
    aucs[t] = auc
    tmp = pd.DataFrame({"sample_id": labeled["sample_id"].values, t: oof_vec})
    oof_all = oof_all.merge(tmp, on="sample_id", how="left")
    test_pred[t] = test_col

# -----------------------------
# SUMMARY + SAVE
# -----------------------------
mean_auc = float(np.mean(list(aucs.values())))
print("\nPer-target AUCs:", {k: round(v, 5) for k, v in aucs.items()})
print("Mean AUC:", mean_auc)

test_pred.to_csv("submission_ohe_levo_cipro_amoxhybrid.csv", index=False)
print("Wrote submission_ohe_levo_cipro_amoxhybrid.csv")

# sanity checks
assert test_pred.shape == (1000, 9)
vals = test_pred.drop(columns=["sample_id"]).values
assert np.isfinite(vals).all()
assert ((vals >= 0) & (vals <= 1)).all()



Training target: Ampicillin
  OOF AUC (Ampicillin): 0.92482   (labeled=3332)

Training target: Levofloxacin
  OOF AUC (Species-specific Levofloxacin): 0.84361   (labeled=3271)

Training target: Ciprofloxacin
  OOF AUC (Species-specific Ciprofloxacin): 0.84789   (labeled=3274)

Training target: Imipenem
  OOF AUC (Imipenem): 0.98916   (labeled=3249)

Training target: Amoxicillin_Clavulanic_acid
  OOF AUC (Hybrid Amoxicillin_Clavulanic_acid): 0.71747   (labeled=1921)

Training target: Ertapenem
  OOF AUC (Ertapenem): 0.98864   (labeled=3360)

Training target: Cefotaxime
  OOF AUC (Cefotaxime): 0.92940   (labeled=3357)

Training target: Cefuroxime
  OOF AUC (Cefuroxime): 0.94280   (labeled=3327)

Per-target AUCs: {'Ampicillin': np.float64(0.92482), 'Levofloxacin': np.float64(0.84361), 'Ciprofloxacin': np.float64(0.84789), 'Imipenem': np.float64(0.98916), 'Amoxicillin_Clavulanic_acid': np.float64(0.71747), 'Ertapenem': np.float64(0.98864), 'Cefotaxime': np.float64(0.9294), 'Cefuroxime': n